# Sesión 4 - Ejercicio: Joins, Filtering Joins y Regex con Polars

Trabajamos con tres archivos:
- `test_data.xlsx`: registro original (20 alumnos)
- `nuevos_alumnos.xlsx`: incorporaciones tardías (8 alumnos)
- `viaje.csv`: postulantes al viaje de fin de curso

Ejecuta las celdas en orden. Cada ejercicio construye sobre el anterior.

## Bloque 0 ¡Recordatorio!: carga, limpieza y variables de asistencia


### Ejercicio 0.1 Carga de archivos

Carga el archivo `test_data.xlsx` con sus cinco hojas y el archivo `nuevos_alumnos.xlsx` con sus tres hojas. Nombra las variables según el patrón `df_<nombre_hoja>` para las originales y `df_new_<nombre_hoja>` para las nuevas (pueden llamarse como sea, pero en los siguientes enunciados asumo que usan `df_<nombre_hoja>` y `df_new_<nombre_hoja>`)

In [ ]:
!pip install polars
!pip install fastexcel

In [ ]:
import polars as pl
import polars.selectors as cs

ruta     = "/content/test_data.xlsx"
ruta_new = "/content/nuevos_alumnos.xlsx"

df_alumnos  =       pl.read_excel(ruta, sheet_name="alumnos")
df_notas    =       pl.read_excel(ruta, sheet_name="notas")
df_asistencia   =   pl.read_excel(ruta, sheet_name="asistencia")
df_profesores   =   pl.read_excel(ruta, sheet_name="profesores")
df_asigna_clase =   pl.read_excel(ruta, sheet_name="asigna_clase")

df_new_alumnos  =     pl.read_excel(ruta_new, sheet_name="alumnos")
df_new_notas    =     pl.read_excel(ruta_new, sheet_name="notas")
df_new_asistencia   = pl.read_excel(ruta_new, sheet_name="asistencia")

### Ejercicio 0.2 Inspeccionar columnas de `df_new_alumnos`

Los datos nuevos llegaron con un formato distinto. Antes de modificar nada, inspeccionamos las columnas de ambas tablas y encontramos las diferencias usando la diferencia de conjuntos.

**Nota rápida sobre sets en Python:**
Un `set` es un conjunto sin orden ni duplicados. La operación `A - B` devuelve los elementos que están en `A` pero no en `B`. Por ejemplo:
```python
set(["a", "b", "c"]) - set(["a", "b"])  # resultado: {'c'}
```

In [90]:
cols_originales = set(df_alumnos.columns)
cols_nuevas     = set(df_new_alumnos.columns)

print("cols_originales - cols_nuevas:",  cols_originales - cols_nuevas)
print("cols_nuevas - cols_originales:", cols_nuevas - cols_originales)

cols_originales - cols_nuevas: {'cod_estudiante', 'nombre', 'edad', 'altura', 'pais', 'mes_nac', 'sexo', 'dia_nac', 'ciudad', 'anio_nac', 'paralelo'}
cols_nuevas - cols_originales: {'SEXO', 'DIA_NAC', 'ALTURA', 'PAIS', 'NOMBRE_APELLIDOS', 'CIUDAD', 'EDAD', 'PARALELO', 'ANIO_NAC', 'MES_NAC', 'COD_ESTUDIANTE'}


Están todos distintos, pero se debe a que están unos los originales están minúsculas y los otros en mayúsculas. Homologuemos eso y comprobemos de nuevo.

### Ejercicio 0.3 Normalizar columnas de `df_new_alumnos`

Las columnas de `df_new_alumnos` vienen en **mayúsculas** y una tiene un nombre distinto (`NOMBRE_APELLIDOS` en lugar de `nombre`). Hacemos dos pasos en cadena:
1. Pasar **todos** los nombres a minúsculas con `.rename()` y una función lambda.
2. Renombrar **a mano** la columna que sigue siendo distinta.

`.rename()` acepta tanto un diccionario `{"viejo": "nuevo"}` como una función que se aplica a cada nombre de columna.

In [91]:
df_new_alumnos = df_new_alumnos.rename(lambda nom_col: nom_col.lower())

# ¿qué no coincide?

print("cols_originales - cols_nuevas:",  cols_originales - set(df_new_alumnos.columns))
print("cols_nuevas - cols_originales:", set(df_new_alumnos.columns) - cols_originales)

cols_originales - cols_nuevas: {'nombre'}
cols_nuevas - cols_originales: {'nombre_apellidos'}


La columna que no coincide hay que renombrarla como en la tabla original

In [ ]:
df_new_alumnos = df_new_alumnos.rename({"_____________": "_____________"})

print("cols_originales - cols_nuevas:",  cols_originales - set(df_new_alumnos.columns))
print("cols_nuevas - cols_originales:", set(df_new_alumnos.columns) - cols_originales)

cols_originales - cols_nuevas: set()
cols_nuevas - cols_originales: set()


Hacemos lo mismo para `df_new_notas` y `df_new_asistencia` ¡solo lowercase, no hay columna con nombre distinto!

In [93]:
df_new_notas = df_new_notas.rename(lambda col: col.lower())
df_new_asistencia = df_new_asistencia.rename(lambda col: col.lower())

# verificamos el cambio
print("nombres modificados de notas:", df_new_notas.columns)
print("nombres modificados de asistencia:", df_new_asistencia.columns)

nombres modificados de notas: ['cod_estudiante', 'nombre', 'materia', 'nota']
nombres modificados de asistencia: ['cod_estudiante', 'nombre', 'a_19950501', 'a_19950502', 'a_19950503', 'a_19950504', 'a_19950505', 'a_19950508', 'a_19950509', 'a_19950510', 'a_19950511', 'a_19950512', 'a_19950515', 'a_19950516', 'a_19950517', 'a_19950518', 'a_19950519', 'a_19950522', 'a_19950523', 'a_19950524', 'a_19950525', 'a_19950526', 'a_19950529', 'a_19950530', 'a_19950531']


### Ejercicio 0.4 Concatenar las tablas

`pl.concat()` apila verticalmente (con `how="vertical"`) dos o más DataFrames con el mismo esquema. El resultado tiene todas las filas de ambas tablas en orden. Notar que al no tener las mismas columnas en asistencia, es necesario usar `how="diagonal"` que llena con null las columnas no comunes.

In [94]:
# Tamaño antes del concat

print("Alumnos:",    df_alumnos.shape)
print("Notas:",      df_notas.shape)
print("Asistencia:", df_asistencia.shape)

Alumnos: (20, 11)
Notas: (80, 4)
Asistencia: (20, 26)


In [ ]:
df_alumnos    = pl.concat([___________  ,   _________], how="vertical")
df_notas      = pl.concat([___________  ,   _________], how="vertical")
df_asistencia = pl.concat([___________  ,   _________], how="diagonal")

# Tamaño después del concat
print("Alumnos:",    df_alumnos.shape)
print("Notas:",      df_notas.shape)
print("Asistencia:", df_asistencia.shape)

Alumnos: (28, 11)
Notas: (112, 4)
Asistencia: (28, 26)


### Ejercicio 0.5 Calcular variables de asistencia

A partir de `df_asistencia`, construimos la tabla `resumen_asistencia` con una fila por alumno. El proceso tiene cuatro pasos:
1. **unpivot** de las columnas `a_*` para pasar de formato wide a long
2. **Recodificar** `X` -> 1 y nulo -> 0
3. **group_by** por alumno
4. **Agregar** % asistencia y variable booleana como True cuando tiene más del 80% de asistencia

In [ ]:
# Paso 1: unpivot
asist_long = df_asistencia._________(
    on=cs.starts_with("a_"),
    index=["cod_estudiante", "nombre"],
    variable_name="fecha",
    value_name="asistio_raw",
)

asist_long

In [ ]:
# Paso 2: recodificar
asist_long = asist_long.with_columns(
    pl.when(pl.col("_________") == "_____")
      .then(pl.lit(1))
      .otherwise(pl.lit(0))
      .alias("asistio")
)

asist_long

cod_estudiante,nombre,dias_asistidos,tasa_asistencia,asistencia_ok
i64,str,i32,f64,bool
18,"""Maria-Josefa Chico""",20,0.869565,true
24,"""Omar Villalobos""",21,0.913043,true
21,"""Sofía Méndez""",22,0.956522,true
15,"""Pascuala Lamas""",15,0.652174,false
13,"""Rayan Macias""",3,0.130435,false
8,"""Gloria Berrocal""",23,1.0,true
20,"""Maria-Gracia Maestre""",14,0.608696,false
9,"""Andreea Centeno""",6,0.26087,false
22,"""Diego Quispe""",20,0.869565,true


In [ ]:
# Pasos 3 y 4: group_by y agregar
resumen_asistencia = (
    asist_long
    .group_by(["cod_estudiante", "nombre"])
    .agg(pl.col("asistio").______().alias("tasa_asistencia"))
    .with_columns((pl.col("tasa_asistencia") > 0.80).alias("asistencia_ok"))
)

resumen_asistencia.head(10)

## Bloque 1 Full join: todos los alumnos con indicador de incorporación tardía


No sabemos si en el nuevo listado nos enviaron alumnos anteriores, por eso hacemos un full join entre los **20 alumnos originales** y los **8 nuevos** para obtener una tabla completa con los 28. Luego creamos la columna `nuevo_estudiante`:
- `True` si el alumno solo aparece en la tabla nueva
- `False` si ya estaba en la original

Después de un full join, las filas que solo existen en la tabla derecha tienen `null` en la columna llave de la tabla izquierda. Usamos eso como señal en el `when/then`.

Guardamos el resultado como `df_todos`.

In [ ]:
# Recargamos solo los originales con otro nombre porque df_alumnos ya tiene los datos trabajados
df_alumnos_orig = pl.read_excel(ruta, sheet_name="alumnos")

df_todos = (
    df_alumnos_orig
    .join(df_new_alumnos, on="cod_estudiante", how="_________", suffix="_nuevo")
    .with_columns(
        pl.when(pl.col("cod_estudiante_nuevo").is_not_null())
          .then(pl.lit(_______))
          .otherwise(pl.lit(_______))
          .alias("nuevo_estudiante")
    )
)

df_todos.group_by(["nuevo_estudiante"]).len(name="total")

nuevo_estudiante,total
bool,u32
false,20
true,8


## Bloque 2 Tabla base para el ejercicio del viaje
Preparamos los datos para el bloque 3 donde vamos a averiguar quién puede irse de viaje con las condiciones:
- tiene permiso firmado
- tiene las cuotas pagadas
- tiene una asistencia mayor al 80%
- tiene una nota media mayor a 6


### Ejercicio 2.1 Cargar `viaje.csv`

El archivo usa `@` como separador de columnas y encoding UTF8. Hay que especificar ambos parámetros explícitamente al leerlo.

In [ ]:
df_viaje = pl.read_csv("viaje.csv", separator="______", encoding="_______")
df_viaje.head()

cod_estudiante,permiso_firmado,cuotas_pagadas
i64,bool,str
1,true,"""cumplido"""
2,true,"""SI"""
3,false,"""no"""
4,true,"""afirmativo"""
5,true,"""sí"""


### Ejercicio 2.2 Normalizar `cuotas_pagadas` con regex

La columna `cuotas_pagadas` tiene valores semánticamente equivalentes con grafía distinta: `si`, `SI`, `Si`, `afirmativo`, `cumplido`, `positivo`, `no`, `NO`, `negativo`.

Creamos la columna booleana `cuotas_ok` usando `str.contains()` con un patrón regex que captura todas las variantes positivas en una sola expresión.

- `(?i)` hace la búsqueda insensible a mayúsculas y minúsculas
- `|` significa "o": el patrón se cumple si el valor contiene alguna de las opciones
- `[]` lo que está dentro del paréntesis significa, uno de esos caracteres
- `^` significa que el patrón inicia (no lo pongo si quiero buscar solo como finaliza un patrón)
- `$` significa finaliza 

Ver guía de regex en https://regexone.com/references/python o https://evoldyn.gitlab.io/evomics-2018/ref-sheets/R_strings.pdf página 2*

*la segunda guía es más didáctica pero algunas cosas solo aplican a stringr de R

In [99]:
# primero vemos qué texto único existe

df_viaje.group_by("cuotas_pagadas").len().to_pandas()

,cuotas_pagadas,len
0,negativo,3
1,no,6
2,positivo,1
3,cumplido,4
4,SÍ,1
5,SI,4
6,sí,1
7,si,2
8,afirmativo,3
9,Si,1


In [100]:
df_viaje = df_viaje.with_columns(
    pl.col("cuotas_pagadas")
      .str.contains(r"(?i)^[sacp]")
      #.str.contains(r"(?i)^(s[íi]|afirmativo|cumplido|positivo)$") # o podemos ser más estrictos
      .alias("cuotas_ok")
)

# Verificar que no quedaron valores sin clasificar
df_viaje.group_by(["cuotas_pagadas", "cuotas_ok"]).len().sort("cuotas_ok").to_pandas()

,cuotas_pagadas,cuotas_ok,len
0,no,False,6
1,NO,False,2
2,negativo,False,3
3,positivo,True,1
4,sí,True,1
5,si,True,2
6,afirmativo,True,3
7,SÍ,True,1
8,cumplido,True,4
9,SI,True,4


### Ejercicio 2.3 Nota media por alumno

Calculamos el promedio de notas de cada alumno a partir de `df_notas` (28 alumnos). Guardamos el resultado como `nota_media_alumnos`.

In [ ]:
nota_media_alumnos = (
    df_notas
    .group_by("___________")
    .agg(pl.col("_________").________().alias("nota_media"))
)

nota_media_alumnos.sort("cod_estudiante")

cod_estudiante,nota_media
i64,f64
1,3.8984
2,6.6842
3,2.9127
4,7.7811
5,6.8945
…,…
24,6.4593
25,9.4337
26,9.6301


### Ejercicio 2.4 Tabla base `df_viaje_notas`

Unimos `df_viaje` con `nota_media_alumnos` para agregar la nota media a cada fila. Esta tabla será el punto de partida de los cuatro caminos del Bloque 3.

In [ ]:
df_viaje_notas = (
    df_viaje
    .join(nota_media_alumnos, on="_________", how="left")
)

df_viaje_notas.head(8)

cod_estudiante,permiso_firmado,cuotas_pagadas,cuotas_ok,nota_media
i64,bool,str,bool,f64
1,true,"""cumplido""",true,3.8984
2,true,"""SI""",true,6.6842
3,false,"""no""",false,2.9127
4,true,"""afirmativo""",true,7.7811
5,true,"""sí""",true,6.8945
6,false,"""NO""",false,4.6153
7,true,"""negativo""",false,5.7776
8,true,"""SÍ""",true,5.9703


## Bloque 3 Cuatro caminos al mismo resultado

**Pregunta:** ¿Qué alumnos pueden ir al viaje?

Las condiciones son:
- `permiso_firmado == True`
- `cuotas_ok == True`
- `asistencia_ok == True` (tasa > 80 %)
- `nota_media > 6.0`

Llegamos al mismo resultado de cuatro maneras distintas. Cada una produce una tabla con nombre propio. Al final verificamos que las cuatro son idénticas.

### Camino A Inner join

Lógica: filtramos primero los que **sí cumplen** la asistencia y luego hacemos un inner join para quedarnos solo con las filas que aparecen en ambas tablas.

El inner join descarta automáticamente cualquier alumno que no tenga coincidencia, es decir, cualquier alumno que no esté en `asistencia_cumple`.

In [ ]:
asistencia_cumple = resumen_asistencia.filter(pl.col("asistencia_ok") == _________)

condiciones_cumple = df_viaje_notas.filter(
        (pl.col("permiso_firmado") == ________) & 
        (pl.col("cuotas_ok") == _______) & 
        (pl.col("_________") > 6.0)
    )

puede_ir_a = (
    condiciones_cumple
    .join(asistencia_cumple.select("cod_estudiante"), 
          on="___________", 
          how="__________")
    .sort("cod_estudiante")
    .drop("cuotas_pagadas")
)

puede_ir_a

cod_estudiante,permiso_firmado,cuotas_ok,nota_media
i64,bool,bool,f64
2,true,true,6.6842
14,true,true,7.0133
17,true,true,6.6887
18,true,true,7.7194
21,true,true,6.9918
22,true,true,7.7164
24,true,true,6.4593
28,true,true,7.2698


### Camino B Anti join

Lógica: identificamos primero los que **no cumplen** la asistencia y luego usamos un anti join para eliminarlos de `df_viaje_notas`.

El anti join conserva las filas de la izquierda que **no tienen** coincidencia en la derecha. Si la derecha contiene los que no cumplen, el anti join los excluye.

In [ ]:
asistencia_no_cumple = resumen_asistencia.filter(pl.col("asistencia_ok") == ___________)

condiciones_cumple = df_viaje_notas.filter(
        (pl.col("permiso_firmado") == ________) & 
        (pl.col("cuotas_ok") == _______) & 
        (pl.col("_________") > 6.0)
)
        
puede_ir_b = (
    condiciones_cumple
    .join(asistencia_no_cumple.select("cod_estudiante"), 
          on="____________", 
          how="___________")
    .sort("cod_estudiante")
    .drop("cuotas_pagadas")
)

puede_ir_b

cod_estudiante,permiso_firmado,cuotas_ok,nota_media
i64,bool,bool,f64
2,true,true,6.6842
14,true,true,7.0133
17,true,true,6.6887
18,true,true,7.7194
21,true,true,6.9918
22,true,true,7.7164
24,true,true,6.4593
28,true,true,7.2698


### Camino C Left join con filter al final

Lógica: traemos todas las columnas de asistencia mediante un left join y aplicamos todas las condiciones juntas en un único `.filter()` al final.

El left join conserva todas las filas de la izquierda. Las filas sin coincidencia quedan con `null` en las columnas nuevas, pero como filtramos con condiciones estrictas, esos nulls quedan excluidos de todas formas.

In [ ]:
condiciones_cumple = df_viaje_notas.filter(
        (pl.col("permiso_firmado") == ________) & 
        (pl.col("cuotas_ok") == _______) & 
        (pl.col("_________") > 6.0)
)

puede_ir_c = (
    condiciones_cumple
    .join(
        resumen_asistencia.select(["cod_estudiante", "asistencia_ok"]),
        on="___________",
        how="__________",
    )
    .filter((pl.col("____________") == True))
    .sort("cod_estudiante")
    .drop("cuotas_pagadas")
)

puede_ir_c

cod_estudiante,permiso_firmado,cuotas_ok,nota_media,asistencia_ok
i64,bool,bool,f64,bool
2,true,true,6.6842,true
14,true,true,7.0133,true
17,true,true,6.6887,true
18,true,true,7.7194,true
21,true,true,6.9918,true
22,true,true,7.7164,true
24,true,true,6.4593,true
28,true,true,7.2698,true


### Camino D Semi join

Lógica: filtramos `resumen_asistencia` para quedarnos con los que cumplen asistencia y usamos un semi join para filtrar `df_viaje_notas` por existencia en esa tabla.

El semi join conserva las filas de la izquierda que tienen coincidencia en la derecha, pero **no agrega columnas** nuevas. A diferencia del inner join, el resultado no crece en columnas, solo se filtra.

In [ ]:
asistencia_cumple = resumen_asistencia.filter(pl.col("asistencia_ok") == True)

condiciones_cumple = df_viaje_notas.filter(
        (pl.col("permiso_firmado") == True) & 
        (pl.col("cuotas_ok") == True) & 
        (pl.col("nota_media") > 6.0)
    )

puede_ir_d = (
    condiciones_cumple
    .join(
        asistencia_cumple,
        on="__________",
        how="_________",
    )
    .sort("cod_estudiante")
    .drop("cuotas_pagadas")
)

puede_ir_d

cod_estudiante,permiso_firmado,cuotas_ok,nota_media
i64,bool,bool,f64
2,true,true,6.6842
14,true,true,7.0133
17,true,true,6.6887
18,true,true,7.7194
21,true,true,6.9918
22,true,true,7.7164
24,true,true,6.4593
28,true,true,7.2698
